In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [3]:
df = pd.read_csv('qoute_dataset.csv')
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [4]:
df.shape

(3038, 2)

In [6]:
quotes = df['quote']
quotes.head()

0    “The world as we have created it is a process ...
1    “It is our choices, Harry, that show what we t...
2    “There are only two ways to live your life. On...
3    “The person, be it gentleman or lady, who has ...
4    “Imperfection is beauty, madness is genius and...
Name: quote, dtype: object

In [7]:
quotes = quotes.str.lower()

In [8]:
import string
translator = str.maketrans(' ',' ',string.punctuation)
quotes = quotes.apply(lambda x: x.translate(translator))

In [9]:
quotes.head()

0    “the world as we have created it is a process ...
1    “it is our choices harry that show what we tru...
2    “there are only two ways to live your life one...
3    “the person be it gentleman or lady who has no...
4    “imperfection is beauty madness is genius and ...
Name: quote, dtype: object

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer                                                                                  # type: ignore

In [11]:
vocab_size = 10000

tok = Tokenizer(num_words = vocab_size)
tok.fit_on_texts(quotes)

In [12]:
word_index = tok.word_index
print(len(word_index))
list(word_index.items())[:10]

8978


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

In [13]:
sequence = tok.texts_to_sequences(quotes)

In [14]:
for i in range(3):
      print(quotes[i])

“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”
“it is our choices harry that show what we truly are far more than our abilities”
“there are only two ways to live your life one is as though nothing is a miracle the other is as though everything is a miracle”


In [15]:
for i in range(3):
      print(sequence[i])

[713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104, 752, 70, 2461]
[947, 7, 70, 871, 373, 9, 433, 21, 19, 465, 14, 294, 52, 54, 70, 3676]
[1337, 14, 53, 201, 714, 3, 81, 15, 36, 37, 7, 29, 329, 93, 7, 5, 1157, 1, 101, 7, 29, 329, 126, 7, 5, 3677]


In [17]:
X = []
y = []

for seq in sequence:
      for i in range(1,len(seq)):
            
            input_seq = seq[:i]
            output_seq = seq[i]
            X.append(input_seq)
            y.append(output_seq)

In [18]:
len(X)

85271

In [19]:
len(y)

85271

In [20]:
max_len = max(len(x) for x in X)
print(max_len)

745


In [24]:
from tensorflow.keras.preprocessing.sequence import pad_sequences                                                                                      # type: ignore
X_padded = pad_sequences(X, maxlen = max_len, padding = 'pre')

In [25]:
y = np.array(y)

In [26]:
X_padded.shape

(85271, 745)

In [ ]:
from tensorflow.keras.utils import to_categorical                                                                                         # type: ignore
y_one_hot = to_categorical(y, num_classes = vocab_size)

In [28]:
y.shape

(85271,)

In [29]:
y_one_hot.shape

(85271, 10000)

In [ ]:
from tensorflow.keras.models import Sequential                                                                                                       # type: ignore
from tensorflow.keras.layers import Embedding, LSTM, Dense ,SimpleRNN                                                                                # type: ignore

In [31]:
embedding_dim = 150
rnn_units = 128

In [32]:
rnn_model = Sequential()

In [35]:
rnn_model.add(Embedding(input_dim = vocab_size, output_dim = embedding_dim, input_length = max_len))
rnn_model.add(SimpleRNN(units = vocab_size))

c:\Users\SANSKAR\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [36]:
rnn_model.compile(optimizer = 'adam', loss = 'categorical_crossentropy', metrics = ['accuracy'])

In [37]:
rnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [38]:
lstm_model = Sequential()
lstm_model.add(Embedding(input_dim = vocab_size, output_dim = embedding_dim, input_length = max_len))
lstm_model.add(LSTM(units = rnn_units))
lstm_model.add(Dense(units = vocab_size, activation = 'softmax'))

c:\Users\SANSKAR\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [39]:
lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [40]:
from tensorflow.keras.models import load_model                                                                                           # type: ignore

In [42]:
lstm_model = load_model('lstm_model.h5')

In [43]:
lstm_model.save('lstm_model.h5')

In [44]:
index_to_word = {}
for word, index in word_index.items():
      index_to_word[index] = word

In [45]:
from tensorflow.keras.preprocessing.text import Tokenizer                                                                                  # type: ignore

In [46]:
def predictor(model, tokenizer, text, max_len):
      text = text.lower()
      
      seq = tokenizer.texts_to_sequences([text])[0]
      seq = pad_sequences([seq], maxlen=max_len, padding = 'pre')
      
      pred = model.predict(seq,verbose = 0)
      pred_index = np.argmax(pred)
      return index_to_word[pred_index] 

In [47]:
seed_text = "what are you"
next_word = predictor(lstm_model, tok, seed_text, max_len)
print(next_word)

worrying


In [50]:
def generate_text(model,tokenizer,seed_text,max_len,n_words):
  for _ in range(n_words):
    next_word = predictor(model,tokenizer,seed_text,max_len)
    
    if next_word == " ":
      break
    seed_text += " " + next_word
    
  return seed_text

In [51]:
seed = "are you a"
generate_text = generate_text(lstm_model, tok, seed, max_len, 10)
print(generate_text)

are you a thousand times i wrote the less if it does not


In [52]:
import pickle
with open("tokenizer.pkl", "wb") as f:
  pickle.dump(tok, f) 

In [53]:
with open("max_len.pkl", "wb") as f:
  pickle.dump(max_len, f)